In [0]:
CATALOG = "olist_ecommerce"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

In [0]:
df_orders = spark.table(f"{BRONZE}.orders")
df_order_items = spark.table(f"{BRONZE}.order_items")
df_products = spark.table(f"{BRONZE}.products")
df_customers = spark.table(f"{BRONZE}.customers")
df_sellers = spark.table(f"{BRONZE}.sellers")
df_payments = spark.table(f"{BRONZE}.payments")
df_reviews = spark.table(f"{BRONZE}.reviews")
df_category_translation = spark.table(f"{BRONZE}.category_translation")

In [0]:
df_orders_clean = df_orders.dropDuplicates(["order_id"])

In [0]:
from pyspark.sql.functions import to_timestamp

df_orders_clean = (
    df_orders_clean
    .withColumn(
        "order_purchase_timestamp",
        to_timestamp("order_purchase_timestamp")
    )
    .withColumn(
        "order_approved_at",
        to_timestamp("order_approved_at")
    )
    .withColumn(
        "order_delivered_carrier_date",
        to_timestamp("order_delivered_carrier_date")
    )
    .withColumn(
        "order_delivered_customer_date",
        to_timestamp("order_delivered_customer_date")
    )
    .withColumn(
        "order_estimated_delivery_date",
        to_timestamp("order_estimated_delivery_date")
    )
)

In [0]:
from pyspark.sql.functions import (
    year,
    month,
    dayofmonth,
    date_format,
    datediff,
    col, 
    when
)

df_orders_clean = (
    df_orders_clean
    .withColumn(
        "purchase_year",
        year("order_purchase_timestamp")
    )
    .withColumn(
        "purchase_month",
        month("order_purchase_timestamp")
    )
    .withColumn(
        "purchase_date",
        date_format(
            "order_purchase_timestamp",
            "yyyy-MM-dd"
        )
    )
)

In [0]:
df_orders_clean = df_orders_clean.withColumn(
    "delivery_delay_days",
    datediff(
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
)

In [0]:
df_orders_clean = df_orders_clean.withColumn(
    "delivery_performance",
    when(
        col("order_delivered_customer_date").isNull(),
        "Not Delivered"
    )
    .when(
        col("delivery_delay_days") > 0,
        "Late"
    )
    .otherwise("On Time")
)

In [0]:
df_order_items.printSchema()

In [0]:
from pyspark.sql.functions import to_timestamp

df_order_items_clean = (
    df_order_items
    .dropDuplicates([
        "order_id",
        "order_item_id"
    ])
    .withColumn(
        "shipping_limit_date",
        to_timestamp("shipping_limit_date")
    )
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))
    .withColumn("total_item_value", col("price") + col("freight_value"))
)

In [0]:
df_order_items_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "olist_ecommerce.silver.order_items_clean"
    )

In [0]:
df_products.printSchema()

In [0]:
df_products_clean = df_products.dropDuplicates(["product_id"])

df_products_clean = (
    df_products_clean
    .withColumn("product_weight_g", col("product_weight_g").cast("double"))
    .withColumn("product_length_cm", col("product_length_cm").cast("double"))
    .withColumn("product_height_cm", col("product_height_cm").cast("double"))
    .withColumn("product_width_cm", col("product_width_cm").cast("double"))
)

In [0]:
df_products_clean = (
    df_products_clean.alias("p")
    .join(
        df_category_translation.alias("t"),
        col("p.product_category_name") ==
        col("t.product_category_name"),
        "left"
    )
    .select(
        col("p.*"),
        col("t.product_category_name_english")
    )
)

In [0]:
df_products_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "olist_ecommerce.silver.products_clean"
    )

In [0]:
df_customers_clean = (df_customers.dropDuplicates(["customer_id"]))

df_customers_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "olist_ecommerce.silver.customers_clean"
    )

In [0]:
df_sellers_clean = (df_sellers.dropDuplicates(["seller_id"]))

df_sellers_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "olist_ecommerce.silver.sellers_clean"
    )

In [0]:
df_payments.printSchema()

In [0]:
df_payments_clean = (
    df_payments
    .withColumn(
        "payment_value",
        col("payment_value").cast("double")
    )
)

df_payments_clean = df_payments_clean.dropDuplicates()

In [0]:
df_payments_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "olist_ecommerce.silver.payments_clean"
    )

In [0]:
df_reviews.printSchema()

In [0]:
df_reviews_clean = df_reviews.dropDuplicates()

In [0]:
df_reviews_clean = df_reviews_clean.withColumn(
    "review_creation_date",
    to_timestamp("review_creation_date")
)

In [0]:
# Criando o Sales Detail
df_sales_detail = (
    df_order_items_clean.alias("oi")
    .join(
        df_orders_clean.alias("o"),
        col("oi.order_id") == col("o.order_id"),
        "inner"
    )
    .join(
        df_products_clean.alias("p"),
        col("oi.product_id") == col("p.product_id"),
        "left"
    )
    .join(
        df_sellers_clean.alias("s"),
        col("oi.seller_id") == col("s.seller_id"),
        "left"
    )
    .join(
        df_customers_clean.alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left"
    )
)

In [0]:
df_sales_detail = df_sales_detail.select(
    col("o.order_id").alias("order_id"),
    col("o.customer_id").alias("customer_id"),
    col("o.order_status").alias("order_status"),
    col("o.order_purchase_timestamp").alias("order_purchase_timestamp"),
    col("o.purchase_year").alias("purchase_year"),
    col("o.purchase_month").alias("purchase_month"),
    col("o.purchase_date").alias("purchase_date"),
    col("o.delivery_delay_days").alias("delivery_delay_days"),
    col("o.delivery_performance").alias("delivery_performance"),

    col("oi.order_item_id").alias("order_item_id"),
    col("oi.product_id").alias("product_id"),
    col("oi.seller_id").alias("seller_id"),
    col("oi.price").alias("price"),
    col("oi.freight_value").alias("freight_value"),
    col("oi.total_item_value").alias("total_item_value"),

    col("p.product_category_name").alias("product_category_name"),
    col("p.product_category_name_english").alias(
        "product_category_name_english"
    ),

    col("c.customer_city").alias("customer_city"),
    col("c.customer_state").alias("customer_state"),

    col("s.seller_city").alias("seller_city"),
    col("s.seller_state").alias("seller_state")
)

In [0]:
display(df_sales_detail)

In [0]:
df_sales_detail.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "olist_ecommerce.silver.sales_detail"
    )

In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM olist_ecommerce.silver.sales_detail;